# 01 — Geometry visualization

Sanity-check the DECAL ECal barrel geometry by parsing the XML directly and confirming that the simulation hits land where we expect: ~30 silicon sampling layers between r=1264 and r=1403 mm, at y-coordinates we can compute analytically from the XML.

By the end you should:
- understand the layer structure (20 thin layers + 10 thick layers),
- visually confirm the simulation hits lie on the predicted layer planes,
- have a top-down view of one silicon layer showing the 100 µm pixel grid.

**Kernel**: `Python (Key4hep)`.


## 1. Parse the XML to compute predicted layer positions

In [ ]:
import os, glob, xml.etree.ElementTree as ET
import uproot, awkward as ak, numpy as np, matplotlib.pyplot as plt

CALOMAPS_HOME = os.environ.get("CALOMAPS_HOME", os.path.expanduser("~/CALOMAPS"))
DATA_BASE = os.environ.get("CALOMAPS_DATA_BASE", os.path.expanduser("~/CALOMAPS-data"))

def parse_sid_value(v, constants):
    if not v: return 0.0
    if v in constants: return parse_sid_value(constants[v], constants)
    s = v.replace('*', ' * ').replace('cm', '10').replace('mm', '1')
    try: return eval(s, {"__builtins__": None}, {})
    except: return float(v)

geom_dir = os.path.join(CALOMAPS_HOME, "geometry")
main_xml = ET.parse(os.path.join(geom_dir, "SiD_TestBeam.xml")).getroot()
consts = {c.get("name"): c.get("value") for c in main_xml.findall(".//constant")}
custom_xml = ET.parse(os.path.join(geom_dir, "my_custom_ecal.xml")).getroot()
det = custom_xml.find(".//detector[@name='ECalBarrel']")

rmin = parse_sid_value(det.find("dimensions").get("rmin"), consts)
print(f"Detector face (rmin): {rmin:.2f} mm")

sensor_y_planes = []
curr_y = rmin
for layer in det.findall("layer"):
    rep = int(layer.get("repeat", 1))
    slices = layer.findall("slice")
    thick = sum(parse_sid_value(s.get("thickness"), consts) for s in slices)
    si_off = 0
    off = 0
    for s in slices:
        t = parse_sid_value(s.get("thickness"), consts)
        if s.get("material") == "Silicon":
            si_off = off + t / 2.0
        off += t
    for _ in range(rep):
        sensor_y_planes.append(curr_y + si_off)
        curr_y += thick

print(f"Total silicon layers: {len(sensor_y_planes)}")
print(f"First 5 layers at y = {[f'{y:.2f}' for y in sensor_y_planes[:5]]} mm")
print(f"Last 5 layers at y  = {[f'{y:.2f}' for y in sensor_y_planes[-5:]]} mm")


## 2. Open one simulation file and compare to predicted layers

In [ ]:
files = sorted(glob.glob(os.path.join(DATA_BASE, "data_spectrum_100um_400GeV", "sim_photons_part*.root")))
filepath = files[0]
print(f"opening {filepath}")

with uproot.open(filepath) as f:
    tree = f["events"]
    x_j = tree["ECalBarrelHits/ECalBarrelHits.position.x"].array()
    y_j = tree["ECalBarrelHits/ECalBarrelHits.position.y"].array()
    z_j = tree["ECalBarrelHits/ECalBarrelHits.position.z"].array()

x_all = ak.flatten(x_j).to_numpy()
y_all = ak.flatten(y_j).to_numpy()
z_all = ak.flatten(z_j).to_numpy()

# Filter to the +Y face only (where the photons enter)
mask = (np.abs(x_all) <= 50) & (np.abs(z_all) <= 50) & (y_all >= 1260) & (y_all <= 1420)
x, y, z = x_all[mask], y_all[mask], z_all[mask]
print(f"Total hits in file:        {len(x_all)}")
print(f"Hits in +Y target volume:  {len(x)}")


## 3. Reverse-engineer the geometry from the hit positions

We can recover both pixel pitch (from adjacent hit spacings) and layer count (from unique y values) directly from the data, and compare to the XML-derived values.


In [ ]:
unique_x = np.unique(np.round(x, 4))
unique_y = np.unique(np.round(y, 4))
unique_z = np.unique(np.round(z, 4))

dx = np.diff(unique_x)
dz = np.diff(unique_z)
pitch_x = np.min(dx[dx > 0]) if len(dx) > 0 else 0
pitch_z = np.min(dz[dz > 0]) if len(dz) > 0 else 0

print(f"Pixel X pitch (from data):  {pitch_x:.5f} mm    (expect 0.1 mm = 100 um)")
print(f"Pixel Z pitch (from data):  {pitch_z:.5f} mm")
print(f"Sensor layers (from data):  {len(unique_y)}      (expect 30 from XML)")
print(f"XML-derived layer y values match data:",
      np.allclose(np.array(sensor_y_planes), unique_y[:len(sensor_y_planes)], atol=0.01))


## 4. Visual diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Side profile (y vs z)
axes[0].scatter(z, y, s=1, alpha=0.3, color="royalblue")
axes[0].set_title("Side profile: y vs z\n(every dot is a pixel hit; horizontal stripes = layers)")
axes[0].set_xlabel("z [mm]")
axes[0].set_ylabel("y (radial) [mm]")
axes[0].grid(True, alpha=0.3)

# Top-down view on the middle silicon layer
target_layer = unique_y[len(unique_y) // 2]
layer_mask = np.abs(y - target_layer) < 0.01
axes[1].scatter(x[layer_mask], z[layer_mask], s=2, alpha=0.8, color="crimson")
axes[1].set_title(f"Top-down hit map on layer y={target_layer:.1f} mm\n(100 um pixel grid should be visible)")
axes[1].set_xlabel("x [mm]")
axes[1].set_ylabel("z [mm]")
axes[1].axis("equal")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## What you confirmed

- 30 sensor layers between r ≈ 1264 and r ≈ 1403 mm, matching the XML
- Pixel pitch of 100 µm (= `ECal_cell_size` in `SiD_TestBeam.xml`)
- Hits cluster along the +Y direction (the beam axis we configured in `sim/run_sim.py`)

**To change the geometry** (e.g. try 50 µm pixels): edit `ECal_cell_size` in [`geometry/SiD_TestBeam.xml`](../geometry/SiD_TestBeam.xml), then re-run the simulation. See [`docs/handbook.md`](../docs/handbook.md) §3 for the cheat sheet.

Next: [`02_data_extraction.ipynb`](02_data_extraction.ipynb) extracts per-event features across all 1000 simulation files into a single .npz for the ML pipeline.
